In [2]:
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

sys.path.append(os.path.abspath(os.path.join('..')))

In [3]:
from src.data_preprocessing import clean_data, get_preprocessing_pipeline, add_features

In [23]:
df = pd.read_csv('../data/raw_data.csv')
df = clean_data(df)
df = add_features(df)

In [24]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

In [30]:
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = [col for col in X.columns if col not in num_features]

In [31]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, recall_score

def objective(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "auto_class_weights": "Balanced",
        "verbose": False,
        "early_stopping_rounds": 50,
        "cat_features": cat_features
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_valid, y_valid))

    preds = model.predict(X_valid)
    score = f1_score(y_valid, preds)

    return score

In [32]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best parameters:", study.best_params)
print("Best F1-score:", study.best_value)

[I 2026-03-01 14:57:39,064] A new study created in memory with name: no-name-591785e6-0452-4695-b724-78a462a5af15
[I 2026-03-01 14:57:47,442] Trial 0 finished with value: 0.6093117408906883 and parameters: {'learning_rate': 0.0016499359115553364, 'depth': 10, 'l2_leaf_reg': 1.9198135777020022, 'random_strength': 1.43609146931728, 'bagging_temperature': 0.7889566898461067}. Best is trial 0 with value: 0.6093117408906883.
[I 2026-03-01 14:57:52,828] Trial 1 finished with value: 0.6068548387096774 and parameters: {'learning_rate': 0.005138784904920967, 'depth': 9, 'l2_leaf_reg': 9.89078064603918, 'random_strength': 9.255149728996132, 'bagging_temperature': 0.45138155139492986}. Best is trial 0 with value: 0.6093117408906883.
[I 2026-03-01 14:57:55,351] Trial 2 finished with value: 0.616956077630235 and parameters: {'learning_rate': 0.01184346742955498, 'depth': 4, 'l2_leaf_reg': 3.9806172662594883, 'random_strength': 9.340606625444309, 'bagging_temperature': 0.8300794363804264}. Best is t

Best parameters: {'learning_rate': 0.05207606815784399, 'depth': 6, 'l2_leaf_reg': 2.5027870376318777, 'random_strength': 6.151624386797369, 'bagging_temperature': 0.8692557531909096}
Best F1-score: 0.6239495798319328


In [34]:
final_cat_params = study.best_params.copy()
final_cat_params['auto_class_weights'] = 'Balanced'
final_cat_params['iterations'] = 1000
final_cat_params['cat_features'] = cat_features


best_model = CatBoostClassifier(**final_cat_params, verbose=100)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_valid)

0:	learn: 0.6730772	total: 4.75ms	remaining: 4.75s
100:	learn: 0.4745406	total: 303ms	remaining: 2.69s
200:	learn: 0.4534755	total: 591ms	remaining: 2.35s
300:	learn: 0.4093080	total: 963ms	remaining: 2.24s
400:	learn: 0.3806103	total: 1.33s	remaining: 1.98s
500:	learn: 0.3576919	total: 1.68s	remaining: 1.68s
600:	learn: 0.3357585	total: 2.06s	remaining: 1.37s
700:	learn: 0.3184085	total: 2.47s	remaining: 1.05s
800:	learn: 0.3008608	total: 2.85s	remaining: 707ms
900:	learn: 0.2851869	total: 3.22s	remaining: 354ms
999:	learn: 0.2731366	total: 3.6s	remaining: 0us


In [35]:
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.76      0.81      1031
           1       0.51      0.72      0.60       371

    accuracy                           0.75      1402
   macro avg       0.70      0.74      0.71      1402
weighted avg       0.78      0.75      0.76      1402



### Why I chose Logistic Regression over CatBoost:

After experimenting with **CatBoost** and optimizing its hyperparameters with **Optuna**, I decided to return to **Logistic Regression** for the final model.

**Reasons for this choice:**
1.  **Higher Recall:** Logistic Regression achieved a **Recall of 0.81**, while the tuned CatBoost only reached **0.72**. In a churn task, it is more important to catch as many potential leavers as possible.
2.  **Dataset Size:** With about 7k rows, the dataset is relatively small. In this case, a complex model like CatBoost can overfit or simply not provide enough benefit over a well-tuned linear model.
3.  **Interpretability:** Logistic Regression gives clear coefficients (weights), making it easy to explain to the business exactly *why* a customer is at risk.
4.  **Simplicity (Occam's Razor):** Since the simpler model performed better on our key metric, it is the more reliable choice for production.